# Lab 05 — Image Segmentation (Tasks 1 – 10)
One code cell per task. Run **Setup** first and set `PATHS` to the images from the
*Image Link – Lab 05 Tasks* sheet (sudoku.png, coins, smarties.png, brain MRI, dog.jpeg).
Missing files fall back to synthetic images with the same character, so every cell runs.
Every cell saves its final output to `output/` and prints the parameters + written answers.
**Parameters were chosen on the synthetic images — retune them on the real ones.**

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATHS = {
    "sudoku": "sudoku.png",       # Tasks 1, 2
    "coins": "coins.jpg",         # Tasks 3, 7, 8
    "smarties": "smarties.png",   # Tasks 4, 5, 10
    "brain": "brain_mri.jpg",     # Task 6
    "dog": "dog.jpeg",            # Task 9
}
os.makedirs("output", exist_ok=True)


def show(imgs, titles, cols=4, cmap="gray"):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(4.3 * cols, 3.9 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap=cmap)
        plt.title(t, fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.show()


def synth(kind):
    rng = np.random.default_rng(sum(map(ord, kind)))
    if kind == "sudoku":
        img = np.full((450, 450), 225, np.float64)
        for i in range(10):
            t = 4 if i % 3 == 0 else 1
            cv2.line(img, (25 + i * 44, 25), (25 + i * 44, 421), 20, t)
            cv2.line(img, (25, 25 + i * 44), (421, 25 + i * 44), 20, t)
        for r in range(9):
            for c in range(9):
                if rng.random() < 0.45:
                    cv2.putText(img, str(rng.integers(1, 10)), (36 + c * 44, 60 + r * 44),
                                cv2.FONT_HERSHEY_SIMPLEX, 1.0, 25, 2)
        yy, xx = np.mgrid[0:450, 0:450]
        light = 1.1 - 0.9 * (xx + 0.6 * yy) / 720                     # bright top-left -> dark bottom-right
        img = img * light + rng.normal(0, 4, img.shape)
        return cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    if kind == "coins":
        img = np.full((380, 520, 3), 70, np.uint8)
        for (x, y, r) in [(100, 100, 48), (192, 108, 46), (300, 95, 40), (110, 260, 44),
                          (200, 270, 50), (292, 268, 44), (420, 180, 55), (430, 310, 38)]:
            cv2.circle(img, (x, y), r, (185, 185, 185), -1)
            cv2.circle(img, (x, y), r - 7, (150, 150, 150), 2)
        return np.clip(img + rng.normal(0, 7, img.shape), 0, 255).astype(np.uint8)
    if kind == "smarties":
        img = np.full((400, 560, 3), (235, 235, 235), np.uint8)
        cols = [(0, 220, 255), (0, 0, 220), (40, 180, 40), (220, 80, 30), (0, 140, 255), (160, 40, 160)]
        for i in range(22):
            x, y = int(rng.integers(50, 510)), int(rng.integers(50, 350))
            c = cols[i % len(cols)]
            cv2.circle(img, (x, y), 32, c, -1)
            cv2.circle(img, (x - 9, y - 9), 9, tuple(min(255, v + 60) for v in c), -1)  # highlight
            cv2.ellipse(img, (x + 6, y + 8), (24, 18), 0, 0, 180, tuple(int(v * 0.6) for v in c), 4)  # shading
        return cv2.GaussianBlur(img, (3, 3), 0)
    if kind == "brain":
        img = np.zeros((360, 300), np.float64)
        cv2.ellipse(img, (150, 180), (125, 160), 0, 0, 360, 200, -1)   # skull
        cv2.ellipse(img, (150, 180), (112, 147), 0, 0, 360, 110, -1)   # brain matter
        cv2.ellipse(img, (125, 170), (18, 45), 15, 0, 360, 35, -1)     # ventricles
        cv2.ellipse(img, (175, 170), (18, 45), -15, 0, 360, 35, -1)
        cv2.circle(img, (200, 250), 28, 165, -1)                       # bright lesion
        cv2.circle(img, (90, 110), 15, 160, -1)                        # similar-intensity spot elsewhere
        img = cv2.GaussianBlur(img, (5, 5), 0) + rng.normal(0, 5, img.shape)
        return cv2.cvtColor(np.clip(img, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    if kind == "dog":
        img = np.zeros((360, 480, 3), np.uint8)
        img[:150] = (230, 180, 120); img[150:] = (60, 150, 70)          # sky / grass
        cv2.ellipse(img, (240, 220), (110, 70), 0, 0, 360, (40, 90, 150), -1)  # body
        cv2.circle(img, (350, 160), 50, (40, 90, 150), -1)                # head
        cv2.circle(img, (365, 150), 8, (20, 20, 20), -1)
        cv2.ellipse(img, (330, 120), (14, 30), 20, 0, 360, (20, 50, 90), -1)   # ear
        cv2.circle(img, (80, 60), 35, (80, 230, 250), -1)                 # sun
        return np.clip(img + rng.normal(0, 10, img.shape), 0, 255).astype(np.uint8)


def load(key):
    img = cv2.imread(PATHS[key])
    if img is None:
        print(f"'{PATHS[key]}' not found -> synthetic {key} image")
        img = synth(key)
    return img


print("Setup done | OpenCV", cv2.__version__)

In [ ]:
# ===================== TASK 1 — Document under uneven lighting: global vs adaptive =====================
gray = cv2.cvtColor(load("sudoku"), cv2.COLOR_BGR2GRAY)                 # 1. grayscale
Ts = [60, 100, 140]                                                     # 2. three global thresholds
globals_ = [cv2.threshold(gray, T, 255, cv2.THRESH_BINARY)[1] for T in Ts]
# 4. adaptive on the same image (local T = Gaussian-weighted mean of 11x11 block - C)
adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 15, 12)

# 3/5. compare: fraction of the image that is "ink" (black) in the left vs right halves
def ink(mask):
    h, w = mask.shape
    return (mask[:, :w // 2] == 0).mean() * 100, (mask[:, w // 2:] == 0).mean() * 100
rows = [{"method": f"global T={T}", "ink% left": round(ink(m)[0], 1), "ink% right": round(ink(m)[1], 1)}
        for T, m in zip(Ts, globals_)]
rows.append({"method": "adaptive G b=15 C=12", "ink% left": round(ink(adaptive)[0], 1),
             "ink% right": round(ink(adaptive)[1], 1)})
print(pd.DataFrame(rows).to_string(index=False))
print("(Real text covers only a few % of the page; a large ink% on one side means that side failed.)")

show([gray] + globals_ + [adaptive], ["Original"] + [f"Global T={T}" for T in Ts] + ["Adaptive (Gaussian, b=15, C=12)"], cols=5)
cv2.imwrite("output/task1_adaptive.png", adaptive)
print("""6. Failing regions: low T loses the characters in the dark (bottom-right) area, because
   text AND paper there are below T and become one class; high T turns the dark paper black.
   Middle T is a compromise that still fails at one end of the gradient.
7. Selected: adaptive Gaussian, blockSize=15 (about one character wide), C=12 (removes paper noise).
WHY A SINGLE THRESHOLD STRUGGLES: global thresholding assumes the same intensity separates
text from background everywhere. With uneven light the paper on the dark side is darker than
the ink on the bright side, so no single T can be correct for both regions.""")

In [ ]:
# ===================== TASK 2 — Choosing the adaptive strategy (blockSize, C, method) =====================
gray = cv2.cvtColor(load("sudoku"), cv2.COLOR_BGR2GRAY)
blocks, Cs = [5, 15, 51], [2, 7, 15]
methods = {"Mean": cv2.ADAPTIVE_THRESH_MEAN_C, "Gauss": cv2.ADAPTIVE_THRESH_GAUSSIAN_C}
results, titles, stats = [gray], ["Original"], []
for mname, m in methods.items():
    for b in blocks:
        for C in Cs:
            out = cv2.adaptiveThreshold(gray, 255, m, cv2.THRESH_BINARY, b, C)   # b must be odd
            n_blobs = cv2.connectedComponents(255 - out)[0] - 1                  # black specks/blobs
            stats.append({"method": mname, "blockSize": b, "C": C,
                          "ink %": round((out == 0).mean() * 100, 2), "black blobs": n_blobs})
            results.append(out); titles.append(f"{mname} b={b} C={C}")
df = pd.DataFrame(stats)
print(df.to_string(index=False))
show(results, titles, cols=5)
pick = df.sort_values(["black blobs", "ink %"], ascending=[True, False]).iloc[0]
print(f"\nFewest specks: {pick.method} blockSize={pick.blockSize} C={pick.C} "
      "(confirm in the figure that strokes are still unbroken)")
best = cv2.adaptiveThreshold(gray, 255, methods[pick.method], cv2.THRESH_BINARY, int(pick.blockSize), int(pick.C))
cv2.imwrite("output/task2_best_adaptive.png", best)
print("""1. Block too small (5): the window often contains only background or only part of a stroke, so
   the local mean follows noise -> speckle noise and hollow/broken characters.
2. Block too large (51): the local mean averages over a big area, approaching a global threshold
   again; the illumination gradient inside the window causes dark patches.
3. Increasing C lowers each local threshold (T = mean - C), so fewer pixels become black:
   noise disappears, but thin strokes start to break at high C.
4. Cleanest: the combination printed above — a block of about one character (~15 px) with a
   fairly high C removes the speckle while keeping the strokes.
5. Gaussian usually performs better: it weights the centre pixels more, giving smoother edges and
   fewer isolated specks than the flat mean. (Check the 'black blobs' column on your image.)""")

In [ ]:
# ===================== TASK 3 — Quality control with Otsu's thresholding =====================
img = load("coins")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)                           # 1
variants = {
    "original": gray,
    "low contrast (a=0.6, b=40)": cv2.convertScaleAbs(gray, alpha=0.6, beta=40),
    "Gaussian blur 5x5": cv2.GaussianBlur(gray, (5, 5), 0),
    "brighter (b=+40)": cv2.convertScaleAbs(gray, alpha=1.0, beta=40),
}
rows, masks = [], {}
for name, g in variants.items():
    T, mask = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # 3. T found automatically
    rows.append({"variant": name, "Otsu T": T, "foreground %": round((mask > 0).mean() * 100, 1)})
    masks[name] = (g, T, mask)
print(pd.DataFrame(rows).to_string(index=False))                       # 4 & 6

plt.figure(figsize=(15, 3.6 * len(variants)))
for i, (name, (g, T, mask)) in enumerate(masks.items()):
    plt.subplot(len(variants), 3, 3 * i + 1); plt.imshow(g, cmap="gray"); plt.title(f"{name}"); plt.axis("off")
    plt.subplot(len(variants), 3, 3 * i + 2); plt.hist(g.ravel(), 256, range=(0, 256))
    plt.axvline(T, color="r"); plt.title(f"Histogram, Otsu T = {T:.0f}")
    plt.subplot(len(variants), 3, 3 * i + 3); plt.imshow(mask, cmap="gray"); plt.title("Otsu mask"); plt.axis("off")
plt.tight_layout(); plt.show()
cv2.imwrite("output/task3_otsu.png", masks["original"][2])
print("""5/6. The chosen threshold moves with the image: lowering contrast or brightening shifts the two
histogram peaks, and Otsu shifts T with them, while the mask stays nearly identical.
WHY OTSU IS USEFUL: it tests every possible T and picks the one that maximises the between-class
variance w0*w1*(mu0-mu1)^2, i.e. the best split of a bimodal histogram. The programmer does not
have to guess T, and it adapts automatically when lighting/contrast changes between images.
(Assumption: two classes, roughly bimodal histogram.)""")

In [ ]:
# ===================== TASK 4 — Sorting objects by colour (HSV) =====================
img = load("smarties")                                                 # 1. colour
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)                             # 2. HSV
print("HSV of pure yellow BGR(0,220,255):", cv2.cvtColor(np.uint8([[[0, 220, 255]]]), cv2.COLOR_BGR2HSV)[0, 0])

# 3/5. experiments with lower/upper bounds (OpenCV H is 0-179)
too_strict = cv2.inRange(hsv, np.array([24, 230, 240]), np.array([26, 255, 255]))
tests = {
    "H 24-26, S>230, V>240 (too strict)": too_strict,
    "H 20-35, S>150, V>150": cv2.inRange(hsv, np.array([20, 150, 150]), np.array([35, 255, 255])),
    "H 18-38, S>80, V>80 (final)": cv2.inRange(hsv, np.array([18, 80, 80]), np.array([38, 255, 255])),
    "H 10-45, S>40 (too loose)": cv2.inRange(hsv, np.array([10, 40, 40]), np.array([45, 255, 255])),
}
final = tests["H 18-38, S>80, V>80 (final)"]                          # 4. binary mask
k = np.ones((5, 5), np.uint8)
final_clean = cv2.morphologyEx(cv2.morphologyEx(final, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
extracted = cv2.bitwise_and(img, img, mask=final_clean)                # 6. extract
for name, m in tests.items():
    print(f"{name:38s}: {int((m > 0).sum()):6d} pixels, {cv2.connectedComponents(m)[0] - 1} blobs")
show([img] + list(tests.values()) + [final_clean, extracted],
     ["Original"] + list(tests.keys()) + ["Final + open/close", "Extracted yellow"], cols=4)
cv2.imwrite("output/task4_yellow_mask.png", final_clean)
cv2.imwrite("output/task4_yellow_extracted.png", extracted)
print("""WHY THE RESTRICTIVE MASK FAILS: a real object is not one colour. Shading lowers V, highlights
lower S, and the hue drifts slightly at edges, so a narrow H/S/V box keeps only the few pixels that
match exactly -> holes and missing parts. The final range widens S and V (which carry the lighting)
while keeping H narrow enough to exclude orange (H~10-17) and green (H~35-85).""")

In [ ]:
# ===================== TASK 5 — Boundary of a manufactured part (Canny + hysteresis) =====================
img = load("smarties")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 1.4)
pairs = [(50, 100), (50, 180), (50, 260)]                              # low fixed, high increased
edges = [cv2.Canny(blur, lo, hi) for lo, hi in pairs]
for (lo, hi), e in zip(pairs, edges):
    print(f"Canny low={lo}, high={hi}: {np.count_nonzero(e)} edge pixels")

# Visualise strong vs weak candidates for the middle pair (gradient magnitude vs thresholds)
gx = cv2.Sobel(blur, cv2.CV_64F, 1, 0); gy = cv2.Sobel(blur, cv2.CV_64F, 0, 1)
mag = np.hypot(gx, gy)
lo, hi = pairs[1]
vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR) // 3
vis[(mag >= lo) & (mag < hi)] = (0, 200, 255)       # weak candidates (orange)
vis[mag >= hi] = (0, 0, 255)                         # strong (red)
kept = edges[1] > 0
missing = cv2.bitwise_and(edges[0], cv2.bitwise_not(edges[2]))   # in low-high result but lost at high
show([gray] + edges + [vis, missing],
     ["Original"] + [f"Edges low={l} high={h}" for l, h in pairs] +
     ["Strong (red) / weak (orange) candidates", "Edges lost when high: 100 -> 260"], cols=3)
cv2.imwrite("output/task5_canny.png", edges[1])
print("""Strong edges: gradient > high (red) — object outlines with large contrast, always kept.
Weak edges: low <= gradient < high (orange) — kept ONLY if connected to a strong edge (hysteresis).
Missing edges: raising high to 260 leaves fewer strong seeds, so low-contrast boundaries (e.g.
yellow on the light background) disappear (last panel).
Unwanted edges: low thresholds pick up highlights and shading lines inside the objects.
Raising HIGH -> fewer seeds -> fewer, cleaner but incomplete edges. Lowering LOW -> more weak
pixels can attach to seeds -> more continuous but noisier contours.""")

In [ ]:
# ===================== TASK 6 — Region growing in a brain MRI =====================
mri = cv2.cvtColor(load("brain"), cv2.COLOR_BGR2GRAY)                 # 1


def region_growing(img, seed, thresh, connectivity=8):
    """seed = (x, y). Adds neighbours whose |I - I_seed| <= thresh."""
    h, w = img.shape
    mask = np.zeros((h, w), np.uint8)
    seed_val = int(img[seed[1], seed[0]])             # int: avoid uint8 wrap-around
    nbrs = [(1, 0), (-1, 0), (0, 1), (0, -1)]
    if connectivity == 8:
        nbrs += [(1, 1), (1, -1), (-1, 1), (-1, -1)]
    stack = [seed]; mask[seed[1], seed[0]] = 255
    while stack:
        x, y = stack.pop()
        for dx, dy in nbrs:
            nx, ny = x + dx, y + dy
            if 0 <= nx < w and 0 <= ny < h and mask[ny, nx] == 0 \
                    and abs(int(img[ny, nx]) - seed_val) <= thresh:
                mask[ny, nx] = 255
                stack.append((nx, ny))
    return mask


seed = (200, 250)                                     # 2. inside the lesion (hover in plt.imshow to pick)
thresholds = [10, 25, 60]
masks_T = [region_growing(mri, seed, T) for T in thresholds]          # 3-5
seeds = [(200, 250), (150, 230), (125, 170)]                          # lesion, brain matter, ventricle
masks_S = [region_growing(mri, s, 20) for s in seeds]

def overlay(mask, seed):
    o = cv2.cvtColor(mri, cv2.COLOR_GRAY2BGR); o[mask > 0] = (0, 0, 255)
    cv2.circle(o, seed, 4, (0, 255, 0), -1); return o

for T, m in zip(thresholds, masks_T):
    print(f"seed {seed}, T={T}: {int((m > 0).sum())} pixels")
for s, m in zip(seeds, masks_S):
    print(f"seed {s} (I={mri[s[1], s[0]]}), T=20: {int((m > 0).sum())} pixels")
show([mri] + [overlay(m, seed) for m in masks_T],
     ["MRI"] + [f"Seed {seed}, T={T}" for T in thresholds], cols=4)
show([overlay(m, s) for s, m in zip(seeds, masks_S)], [f"Seed {s}, T=20" for s in seeds], cols=3)
cv2.imwrite("output/task6_region_mask.png", masks_T[1])
print("""Parameter study: small T stops early (noise breaks connectivity); medium T fills the lesion;
large T leaks through the weak boundary into neighbouring tissue of similar intensity.
WHY THE SEED MATTERS: the region is defined relative to the seed's intensity AND must be
connected to the seed. A different seed has a different reference value and starts inside a
different connected structure, so it grows into a completely different region (lesion vs
brain matter vs ventricle). Note the similar-intensity spot at top-left is NOT included from the
lesion seed — unlike a global threshold, region growing requires connectivity.""")

In [ ]:
# ===================== TASK 7 — Separating touching coins (marker-based watershed) =====================
img = load("coins")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)                                          # 1 preprocessing
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)     # 2 thresholding
if thresh[0, 0] == 255:                      # background came out white -> coins must be white
    thresh = cv2.bitwise_not(thresh)
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)        # 3 noise removal
sure_bg = cv2.dilate(opening, kernel, iterations=3)                              # 4 sure background
dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)                            # 5 distance transform
_, sure_fg = cv2.threshold(dist, 0.6 * dist.max(), 255, 0)                       # 6 sure foreground
sure_fg = np.uint8(sure_fg)
unknown = cv2.subtract(sure_bg, sure_fg)                                         # 7 unknown region
n, markers = cv2.connectedComponents(sure_fg)                                    # 8 marker labelling
markers = markers + 1                        # background = 1 (0 is reserved for "unknown")
markers[unknown == 255] = 0
markers = cv2.watershed(img, markers)                                            # 9 watershed (3-ch img)
result = img.copy(); result[markers == -1] = (0, 0, 255)                         # 10 boundaries
naive = cv2.connectedComponents(opening)[0] - 1
print(f"Naive connected regions: {naive} | watershed coins: {n - 1}")
dist_vis = cv2.normalize(dist, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
show([img, thresh, sure_bg, dist_vis, sure_fg, unknown, result],
     ["Original", "Threshold (Otsu)", "Sure background", "Distance transform",
      "Sure foreground (0.6*max)", "Unknown region", f"Watershed: {n - 1} coins"])
cv2.imwrite("output/task7_watershed.png", result)
print("Visual check: red boundary lines should run through every contact point between touching coins.")

In [ ]:
# ===================== TASK 8 — Tuning watershed: distance-transform threshold =====================
img = load("coins")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, th = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
if th[0, 0] == 255:
    th = cv2.bitwise_not(th)
k3 = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(th, cv2.MORPH_OPEN, k3, iterations=2)
sure_bg = cv2.dilate(opening, k3, iterations=3)
dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)

rows, outs, titles = [], [], []
for i, frac in enumerate([0.2, 0.4, 0.6, 0.8], 1):
    _, fg = cv2.threshold(dist, frac * dist.max(), 255, 0); fg = np.uint8(fg)
    n, mk = cv2.connectedComponents(fg)
    mk = mk + 1; mk[cv2.subtract(sure_bg, fg) == 255] = 0
    mk = cv2.watershed(img.copy(), mk)
    regions = len(np.unique(mk)) - 2                 # minus boundary (-1) and background (1)
    o = img.copy(); o[mk == -1] = (0, 0, 255)
    outs.append(o); titles.append(f"Exp {i}: {frac}*max -> {regions} regions")
    rows.append({"Experiment": i, "Distance threshold": f"{frac}*max", "Foreground markers": n - 1,
                 "Separated regions": regions})
df = pd.DataFrame(rows)
true_count = None if os.path.exists(PATHS["coins"]) else 8     # synthetic tray has 8 coins
df["Observed result"] = [("merged" if r < true_count else "correct" if r == true_count else "over-split")
                         if true_count else "inspect visually" for r in df["Separated regions"]]
print(df.to_string(index=False))
show(outs, titles, cols=4)
cv2.imwrite("output/task8_best.png", outs[2])
print("""Low threshold: the sure-foreground blobs of touching coins still touch -> one marker for
several coins -> MERGED. High threshold: only the very centres survive; small coins may lose
their marker entirely (their peak is below the cut) -> missed objects. Mid range (~0.4-0.7 of the
max distance here) gives one marker per coin and the most meaningful separation.""")

In [ ]:
# ===================== TASK 9 — Segmenting an image with K-Means =====================
img = load("dog")                                                     # 1
Z = img.reshape(-1, 3)                                                # 2. one BGR feature vector per pixel
Z = np.float32(Z)                                                     # 3. kmeans needs float32
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
segs, rows = [img], []
for K in [2, 4, 6]:
    compactness, labels, centers = cv2.kmeans(Z, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)  # 4
    centers = np.uint8(centers)
    seg = centers[labels.flatten()].reshape(img.shape)               # 5. pixel -> its centre colour
    segs.append(seg)
    err = np.sqrt(np.mean((img.astype(float) - seg) ** 2))
    shares = np.bincount(labels.flatten(), minlength=K) / labels.size * 100
    rows.append({"K": K, "RMSE vs original": round(err, 1),
                 "colours (orig unique -> K)": f"{len(np.unique(Z, axis=0))} -> {K}",
                 "cluster centres BGR (share %)": "; ".join(f"{tuple(c)} {s:.0f}%" for c, s in zip(centers.tolist(), shares))})
    cv2.imwrite(f"output/task9_kmeans_K{K}.png", seg)
print(pd.DataFrame(rows).to_string(index=False))
show(segs, ["Original", "K=2", "K=4", "K=6"], cols=4)
print("""K=2: strongest simplification — roughly 'bright background (sky)' vs 'everything darker'.
K=4: sky, grass, dog body and dark details become separate regions.
K=6: closest to the original (lowest RMSE); extra clusters split shading/noise within regions.
Larger K -> less colour simplification, smaller visual difference from the original.
K-means uses colour only, so pixels of the same colour in different places share a cluster.""")

In [ ]:
# ===================== TASK 10 — Design a segmentation system for an unknown image =====================
img = load("smarties")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Method 1: Otsu on grayscale
T, m1 = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
# Method 2: colour thresholding — any saturated pixel is a sweet (background is unsaturated)
m2 = cv2.inRange(hsv, np.array([0, 70, 50]), np.array([179, 255, 255]))
m2 = cv2.morphologyEx(m2, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
# Method 3: K-means (K=7) on colour, shown as a label image
Z = np.float32(img.reshape(-1, 3))
_, lab, cen = cv2.kmeans(Z, 7, None, (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2),
                         10, cv2.KMEANS_PP_CENTERS)
m3 = np.uint8(cen)[lab.flatten()].reshape(img.shape)

for name, m in [("Otsu", m1), ("Colour (S>70)", m2)]:
    print(f"{name:14s}: foreground {(m > 0).mean() * 100:.1f}%, objects {cv2.connectedComponents(m)[0] - 1}")
print(f"Otsu T = {T:.0f}")
show([img, m1, m2, m3], ["Original", f"Method 1: Otsu (T={T:.0f})", "Method 2: HSV saturation",
                         "Method 3: K-means K=7"], cols=4)
for i, m in enumerate([m1, m2, m3], 1):
    cv2.imwrite(f"output/task10_method{i}.png", m)
print("""METHOD 1 — Otsu (grayscale)
  Assumption: objects and background form two intensity classes (bimodal histogram).
  Succeeds: dark sweets (red, blue, purple, green) against the light background.
  Fails: yellow/orange sweets have gray levels close to the background -> missing or hollow.
  Key parameter: none directly; pre-blur and the colour->gray conversion decide the result.
METHOD 2 — Colour thresholding (HSV saturation)
  Assumption: targets differ from the background in colour, not just brightness.
  Succeeds: every sweet, including yellow, because the background has very low saturation.
  Fails: highlights (low S) punch small holes; touching sweets merge into one blob.
  Key parameter: the lower S bound.
METHOD 3 — K-means (colour clusters)
  Assumption: regions are compact clusters in BGR space; K is known.
  Succeeds: separates sweets by colour into labelled groups.
  Fails: shading/highlights can form their own clusters, ignores spatial connectivity.
  Key parameter: K.
CONCLUSION: colour thresholding gives the most meaningful object/background regions here because
the image's defining property is that the background is unsaturated while every object is
strongly coloured — intensity alone (Otsu) cannot separate light-coloured sweets.""")